In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Upbit Auto Trader (BTC/ETH/XRP)
- 월 1회 DCA, '빗각' 추세선 돌파, 차트 패턴 신호(간이), 분할 매수/매도, 이메일 데일리 리포트
- 필수: pip install pyupbit python-dotenv pandas numpy requests
- .env 예시:
    UPBIT_ACCESS_KEY=xxxx
    UPBIT_SECRET_KEY=xxxx
    TELEGRAM_BOT_TOKEN=
    TELEGRAM_CHAT_ID=
    SMTP_HOST=smtp.gmail.com
    SMTP_PORT=587
    SMTP_USER=youremail@gmail.com
    SMTP_PASS=app_password
    REPORT_TO=receiver@example.com

⚠️ 면책: 교육용 샘플. 실전 전 백테스트/소액 검증 필수. 손실 책임은 사용자 본인에게 있습니다.
"""

from __future__ import annotations
import os, time, json, math, signal, smtplib, logging, datetime as dt
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import requests
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv
import pyupbit

# =========================
# 0) 전역 설정
# =========================
load_dotenv()
ACCESS_KEY = os.getenv("UPBIT_ACCESS_KEY", "")
SECRET_KEY = os.getenv("UPBIT_SECRET_KEY", "")
TG_TOKEN   = os.getenv("TELEGRAM_BOT_TOKEN", "")
TG_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "")

# 이메일 리포트
SMTP_HOST = os.getenv("SMTP_HOST", "")
SMTP_PORT = int(os.getenv("SMTP_PORT", "587"))
SMTP_USER = os.getenv("SMTP_USER", "")
SMTP_PASS = os.getenv("SMTP_PASS", "")
REPORT_TO = os.getenv("REPORT_TO", "")

assert ACCESS_KEY and SECRET_KEY, "UPBIT API 키(.env)가 필요합니다."

# 전략/데이터 세팅
TICKERS = ["KRW-BTC", "KRW-ETH", "KRW-XRP"]
CANDLE_INTERVAL = "day"  # 'minute1', 'minute5', 'day'
LOOKBACK_DAYS = 365*3
REFETCH_EVERY_MIN = 10

# 빗각 돌파
CROSS_BUY_SIZE_KRW = 10000
CROSS_SELL_RATIO   = 1.00
SLIPPAGE_GUARD_PCT = 0.7
COOLDOWN_MIN       = 15
MAX_POS_KRW        = 2_000_000

# 분할 체결
SPLIT_ENABLED      = True
SPLIT_PARTS        = 4
SPLIT_INTERVAL_SEC = 10

# 월 DCA
DCA_ENABLED    = True
DCA_DAY        = 1
DCA_HOUR       = 9
DCA_MINUTE     = 0
DCA_AMOUNT_KRW = 20000

# 데일리 리포트(21:00 KST)
REPORT_HOUR   = 21
REPORT_MINUTE = 0

# 파일
STATE_FILE   = "state.json"
TRADELOG_CSV = "trades.csv"
LOG_FILE     = "upbit_auto_trader.log"

# 로거
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logging.getLogger().addHandler(logging.StreamHandler())

# =========================
# 유틸
# =========================
def now_kst() -> dt.datetime:
    return dt.datetime.now(dt.timezone(dt.timedelta(hours=9)))

def tg_notify(msg: str):
    if not (TG_TOKEN and TG_CHAT_ID):
        return
    try:
        requests.post(
            f"https://api.telegram.org/bot{TG_TOKEN}/sendMessage",
            json={"chat_id": TG_CHAT_ID, "text": msg},
            timeout=10
        )
    except Exception as e:
        logging.warning(f"[TG] notify failed: {e}")

def load_state() -> Dict:
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_state(state: Dict):
    tmp = STATE_FILE + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, STATE_FILE)

def within_cooldown(last_ts_iso: Optional[str]) -> bool:
    if not last_ts_iso:
        return False
    last = dt.datetime.fromisoformat(last_ts_iso)
    return (now_kst() - last) < dt.timedelta(minutes=COOLDOWN_MIN)

# =========================
# 1) Upbit 객체/잔고/주문
# =========================
UPBIT = pyupbit.Upbit(ACCESS_KEY, SECRET_KEY)

def get_balance(currency: str) -> float:
    bal = UPBIT.get_balances()
    for b in bal:
        if b.get('currency') == currency:
            return float(b.get('balance', 0.0))
    return 0.0

def get_coin_balance(market: str) -> float:
    coin = market.split('-')[1]
    bal = UPBIT.get_balances()
    for b in bal:
        if b.get('currency') == coin:
            return float(b.get('balance', 0.0))
    return 0.0

def _log_trade(side: str, market: str, amount_or_qty: float,
               price: Optional[float]=None, meta: Optional[dict]=None):
    row = {
        'ts': now_kst().isoformat(),
        'side': side,
        'market': market,
        'value_or_qty': amount_or_qty,
        'price': price,
        'meta': json.dumps(meta or {}, ensure_ascii=False),
    }
    header = not os.path.exists(TRADELOG_CSV)
    with open(TRADELOG_CSV, 'a', encoding='utf-8') as f:
        if header:
            f.write('ts,side,market,value_or_qty,price,meta\n')
        f.write(','.join([
            str(row[k]) for k in ['ts', 'side', 'market',
                                  'value_or_qty', 'price', 'meta']
        ]) + '\n')

def market_buy_krw(market: str, krw: float) -> Optional[Dict]:
    if krw <= 5000:
        logging.info(f"[BUY-SKIP] {market} KRW {krw} (too small)")
        return None
    try:
        ord = UPBIT.buy_market_order(market, krw)
        logging.info(f"[BUY] {market} krw={krw} → {ord}")
        tg_notify(f"[BUY] {market} {int(krw)} KRW")
        _log_trade('BUY', market, krw, meta={'split': False})
        return ord
    except Exception as e:
        logging.exception(f"[BUY-ERR] {market}: {e}")
        tg_notify(f"[BUY-ERR] {market}: {e}")
        return None

def market_sell_all(market: str, ratio: float = 1.0) -> Optional[Dict]:
    qty = get_coin_balance(market)
    if qty <= 0:
        logging.info(f"[SELL-SKIP] {market}: no position")
        return None
    qty_to_sell = qty * max(0.0, min(1.0, ratio))
    try:
        ord = UPBIT.sell_market_order(market, qty_to_sell)
        logging.info(f"[SELL] {market} qty={qty_to_sell} → {ord}")
        tg_notify(f"[SELL] {market} qty≈{qty_to_sell:.8f}")
        _log_trade('SELL', market, qty_to_sell, meta={'split': False})
        return ord
    except Exception as e:
        logging.exception(f"[SELL-ERR] {market}: {e}")
        tg_notify(f"[SELL-ERR] {market}: {e}")
        return None

# --- 분할 체결 ---
def split_buy_krw(market: str, total_krw: float):
    if not SPLIT_ENABLED:
        return market_buy_krw(market, total_krw)
    parts = max(1, SPLIT_PARTS)
    each = total_krw / parts
    res = None
    for i in range(parts):
        logging.info(f"[BUY-SPLIT] {market} part {i+1}/{parts}, krw={each}")
        res = market_buy_krw(market, each)
        _log_trade('BUY_SPLIT', market, each, meta={'idx': i+1, 'parts': parts})
        time.sleep(SPLIT_INTERVAL_SEC)
    return res

def split_sell_ratio(market: str, ratio: float = 1.0):
    if not SPLIT_ENABLED:
        return market_sell_all(market, ratio)
    parts = max(1, SPLIT_PARTS)
    each = ratio / parts
    res = None
    for i in range(parts):
        logging.info(f"[SELL-SPLIT] {market} part {i+1}/{parts}, ratio={each}")
        res = market_sell_all(market, each)
        _log_trade('SELL_SPLIT', market, each, meta={'idx': i+1, 'parts': parts})
        time.sleep(SPLIT_INTERVAL_SEC)
    return res

# =========================
# 2) 데이터 & 빗각 추세선
# =========================
def fetch_ohlcv_all(market: str, days: int) -> pd.DataFrame:
    """pyupbit.get_ohlcv를 여러 번 호출해서 days만큼의 일봉 모으기."""
    end = None
    frames = []
    remain = days
    logging.info(f"[OHLCV] fetch start {market} days={days}")
    while remain > 0:
        cnt = min(remain, 200)
        # ✅ 여기 ticker= 로 수정
        df = pyupbit.get_ohlcv(ticker=market, interval=CANDLE_INTERVAL,
                               count=cnt, to=end)
        if df is None or df.empty:
            logging.warning(f"[OHLCV] empty for {market} (to={end})")
            break
        frames.append(df)
        remain -= len(df)
        first_idx = df.index.min()
        end = (first_idx - pd.Timedelta(days=1)).strftime('%Y-%m-%d %H:%M:%S')
        time.sleep(0.15)
    if not frames:
        raise RuntimeError(f"OHLCV 수집 실패: {market}")
    out = pd.concat(frames).sort_index().drop_duplicates()
    out["t"] = (out.index - out.index[0]).total_seconds() / 86400.0
    logging.info(f"[OHLCV] fetched {market} rows={len(out)} "
                 f"from={out.index.min()} to={out.index.max()}")
    return out

def trendline_from_extrema(df: pd.DataFrame) -> Tuple[float, float, pd.Timestamp, pd.Timestamp]:
    low_idx = df['low'].idxmin()
    high_idx = df['high'].idxmax()
    if low_idx < high_idx:
        p0_t, p0_p = df.loc[low_idx,'t'], df.loc[low_idx,'low']
        p1_t, p1_p = df.loc[high_idx,'t'], df.loc[high_idx,'high']
    else:
        p0_t, p0_p = df.loc[high_idx,'t'], df.loc[high_idx,'high']
        p1_t, p1_p = df.loc[low_idx,'t'], df.loc[low_idx,'low']
    m = 0.0 if p1_t == p0_t else (p1_p - p0_p) / (p1_t - p0_t)
    b = p0_p - m * p0_t
    return m, b, low_idx, high_idx

def projected_line_value(m: float, b: float, df: pd.DataFrame, when: pd.Timestamp) -> float:
    t = (when - df.index[0]).total_seconds() / 86400.0
    return m * t + b

def detect_cross(prev_price: float, prev_line: float,
                 cur_price: float, cur_line: float) -> str:
    if prev_price <= prev_line and cur_price > cur_line:
        return 'cross_up'
    if prev_price >= prev_line and cur_price < cur_line:
        return 'cross_down'
    return 'none'

# =========================
# 3) 차트 패턴(간이 버전)
# =========================
@dataclass
class PatternSignal:
    name: str
    direction: str  # 'up'|'down'
    breakout_ts: pd.Timestamp
    target: Optional[float]
    meta: Dict

def swing_points(df: pd.DataFrame, k: int = 2) -> Tuple[List[int], List[int]]:
    highs, lows = [], []
    for i in range(k, len(df)-k):
        if df['high'].iloc[i] == max(df['high'].iloc[i-k:i+k+1]):
            highs.append(i)
        if df['low'].iloc[i] == min(df['low'].iloc[i-k:i+k+1]):
            lows.append(i)
    return highs, lows

def detect_rectangle(df: pd.DataFrame) -> Optional[PatternSignal]:
    window = df.iloc[-60:]
    hi = window['high'].rolling(20).max().iloc[-1]
    lo = window['low'].rolling(20).min().iloc[-1]
    if hi/lo < 1.05:
        last = window.iloc[-1]; prev = window.iloc[-2]
        if prev['close'] <= hi and last['close'] > hi:
            target = last['close'] + (hi - lo)
            return PatternSignal('Rectangle', 'up', window.index[-1],
                                 float(target), {'hi': float(hi), 'lo': float(lo)})
        if prev['close'] >= lo and last['close'] < lo:
            target = last['close'] - (hi - lo)
            return PatternSignal('Rectangle', 'down', window.index[-1],
                                 float(target), {'hi': float(hi), 'lo': float(lo)})
    return None

def detect_double_top_bottom(df: pd.DataFrame) -> Optional[PatternSignal]:
    highs, lows = swing_points(df)
    recent = df.iloc[-120:]
    if len(highs) >= 2:
        h1, h2 = highs[-2], highs[-1]
        p1, p2 = df['high'].iloc[h1], df['high'].iloc[h2]
        if abs(p1-p2)/max(p1,p2) < 0.02:
            neck = df['low'].iloc[min(h1,h2):max(h1,h2)].min()
            if recent['close'].iloc[-2] >= neck and recent['close'].iloc[-1] < neck:
                tgt = neck - (max(p1,p2) - neck)
                return PatternSignal('Double Top', 'down', df.index[-1],
                                     float(tgt), {'neckline': float(neck)})
    if len(lows) >= 2:
        l1, l2 = lows[-2], lows[-1]
        p1, p2 = df['low'].iloc[l1], df['low'].iloc[l2]
        if abs(p1-p2)/max(p1,p2) < 0.02:
            neck = df['high'].iloc[min(l1,l2):max(l1,l2)].max()
            if recent['close'].iloc[-2] <= neck and recent['close'].iloc[-1] > neck:
                tgt = neck + (neck - min(p1,p2))
                return PatternSignal('Double Bottom', 'up', df.index[-1],
                                     float(tgt), {'neckline': float(neck)})
    return None

def detect_triangles(df: pd.DataFrame) -> Optional[PatternSignal]:
    win = df.iloc[-120:]
    highs = win['high'].rolling(5).max()
    lows = win['low'].rolling(5).min()
    up = np.polyfit(np.arange(len(highs.dropna())), highs.dropna(), 1)
    dn = np.polyfit(np.arange(len(lows.dropna())), lows.dropna(), 1)
    m_up, b_up = up[0], up[1]
    m_dn, b_dn = dn[0], dn[1]
    idx = len(win)-1
    res_up = m_up*idx + b_up
    res_dn = m_dn*idx + b_dn
    last_close = win['close'].iloc[-1]
    prev_close = win['close'].iloc[-2]
    height = (highs.max() - lows.min())
    if prev_close <= res_up and last_close > res_up:
        return PatternSignal('Triangle', 'up', win.index[-1],
                             float(last_close + height),
                             {'m_up': m_up, 'm_dn': m_dn})
    if prev_close >= res_dn and last_close < res_dn:
        return PatternSignal('Triangle', 'down', win.index[-1],
                             float(last_close - height),
                             {'m_up': m_up, 'm_dn': m_dn})
    return None

def detect_wedge(df: pd.DataFrame) -> Optional[PatternSignal]:
    win = df.iloc[-120:]
    highs = win['high'].rolling(5).max()
    lows = win['low'].rolling(5).min()
    up = np.polyfit(np.arange(len(highs.dropna())), highs.dropna(), 1)
    dn = np.polyfit(np.arange(len(lows.dropna())), lows.dropna(), 1)
    m_up, m_dn = up[0], dn[0]
    last = win.iloc[-1]
    prev = win.iloc[-2]
    if m_up > 0 and m_dn > 0:  # 상승쐐기 → 하방
        support = (dn[0]*(len(win)-1) + dn[1])
        if prev['close'] >= support and last['close'] < support:
            return PatternSignal('Rising Wedge', 'down', win.index[-1],
                                 float(win['low'].min()), {})
    if m_up < 0 and m_dn < 0:  # 하락쐐기 → 상방
        resist = (up[0]*(len(win)-1) + up[1])
        if prev['close'] <= resist and last['close'] > resist:
            height = float(win['high'].max() - win['low'].min())
            return PatternSignal('Falling Wedge', 'up', win.index[-1],
                                 float(last['close'] + height), {})
    return None

def detect_head_shoulders(df: pd.DataFrame) -> Optional[PatternSignal]:
    highs, _ = swing_points(df, k=3)
    if len(highs) < 3:
        return None
    h1, h2, h3 = highs[-3], highs[-2], highs[-1]
    p1, p2, p3 = df['high'].iloc[h1], df['high'].iloc[h2], df['high'].iloc[h3]
    if p2 > p1*0.98 and p2 > p3*0.98 and p2 > p1 and p2 > p3:
        neck = min(df['low'].iloc[h1:h2].min(), df['low'].iloc[h2:h3].min())
        last2, last1 = df['close'].iloc[-2], df['close'].iloc[-1]
        if last2 >= neck and last1 < neck:
            tgt = neck - (p2 - neck)
            return PatternSignal('Head & Shoulders', 'down', df.index[-1],
                                 float(tgt), {'neckline': float(neck)})
    return None

def detect_flag_pennant(df: pd.DataFrame) -> Optional[PatternSignal]:
    win = df.iloc[-80:]
    pole = win['close'].iloc[-15]
    last = win['close'].iloc[-1]
    if pole == 0:
        return None
    if last/pole > 1.15:
        sub = win.iloc[-20:]
        up = np.polyfit(np.arange(len(sub)), sub['close'].values, 1)
        if up[0] <= 0:
            prev = sub['close'].iloc[-2]
            trend = up[0]*(len(sub)-1) + up[1]
            if prev <= trend and last > trend:
                target = float(last + (last - pole))
                return PatternSignal('Flag/Pennant', 'up', win.index[-1],
                                     target, {})
    return None

def detect_gap(df: pd.DataFrame) -> Optional[PatternSignal]:
    prev_close = df['close'].iloc[-2]
    cur_open = df['open'].iloc[-1]
    if prev_close == 0:
        return None
    if cur_open/prev_close > 1.02:
        return PatternSignal('Gap Up', 'up', df.index[-1], None, {})
    if cur_open/prev_close < 0.98:
        return PatternSignal('Gap Down', 'down', df.index[-1], None, {})
    return None

def detect_cup_handle(df: pd.DataFrame) -> Optional[PatternSignal]:
    win = df.iloc[-160:]
    ma = win['close'].rolling(20).mean()
    if ma.isna().sum() > len(ma) - 10:
        return None
    neck = float(win['close'].rolling(60).max().iloc[-20])
    prev = float(win['close'].iloc[-2])
    last = float(win['close'].iloc[-1])
    if prev <= neck and last > neck:
        height = neck - float(win['close'].rolling(60).min().iloc[-60])
        return PatternSignal('Cup & Handle', 'up', win.index[-1],
                             float(last + height), {'neckline': neck})
    return None

def detect_patterns(df: pd.DataFrame) -> List[PatternSignal]:
    detectors = [
        detect_rectangle,
        detect_double_top_bottom,
        detect_triangles,
        detect_wedge,
        detect_head_shoulders,
        detect_flag_pennant,
        detect_gap,
        detect_cup_handle,
    ]
    signals: List[PatternSignal] = []
    for fn in detectors:
        try:
            sig = fn(df)
            if sig:
                signals.append(sig)
        except Exception as e:
            logging.debug(f"pattern detector {fn.__name__} error: {e}")
    return signals

# =========================
# 4) 전략: 빗각 & 패턴 처리
# =========================
def enforce_risk_limits(market: str, add_krw: float) -> bool:
    cur_price = pyupbit.get_current_price(market)
    if not cur_price:
        return False
    qty = get_coin_balance(market)
    pos_value = qty * float(cur_price)
    return (pos_value + add_krw) <= MAX_POS_KRW

def process_trend_signal(market: str, state: Dict, df: pd.DataFrame):
    m, b, _, _ = trendline_from_extrema(df)
    if len(df) < 2:
        return
    prev_ts, cur_ts = df.index[-2], df.index[-1]
    prev_close, cur_close = float(df['close'].iloc[-2]), float(df['close'].iloc[-1])
    prev_line = projected_line_value(m, b, df, prev_ts)
    cur_line  = projected_line_value(m, b, df, cur_ts)
    cross = detect_cross(prev_close, prev_line, cur_close, cur_line)

    st = state.setdefault(market, {})
    if cross != 'none':
        logging.info(
            f"[TREND] {market} {cross} | "
            f"prev_close={prev_close:.1f}, prev_line={prev_line:.1f}, "
            f"cur_close={cur_close:.1f}, cur_line={cur_line:.1f}"
        )

    if cross == 'cross_up':
        if within_cooldown(st.get('last_trade_at')):
            logging.info(f"[TREND] {market} cross_up but cooldown")
        elif not enforce_risk_limits(market, CROSS_BUY_SIZE_KRW):
            logging.info(f"[TREND] {market} cross_up but risk limit")
        else:
            krw = get_balance('KRW')
            if krw >= CROSS_BUY_SIZE_KRW:
                if split_buy_krw(market, CROSS_BUY_SIZE_KRW):
                    st['last_side'] = 'long'
                    st['last_trade_at'] = now_kst().isoformat()
                    st['last_reason'] = 'trend_cross_up'

    elif cross == 'cross_down':
        if within_cooldown(st.get('last_trade_at')):
            logging.info(f"[TREND] {market} cross_down but cooldown")
        else:
            qty = get_coin_balance(market)
            if qty > 0:
                if split_sell_ratio(market, CROSS_SELL_RATIO):
                    st['last_side'] = 'flat'
                    st['last_trade_at'] = now_kst().isoformat()
                    st['last_reason'] = 'trend_cross_down'

    st['trendline'] = {'m': m, 'b': b, 'updated': now_kst().isoformat()}

def process_pattern_signals(market: str, state: Dict, df: pd.DataFrame):
    signals = detect_patterns(df)
    if not signals:
        return
    st = state.setdefault(market, {})
    # 가장 최근 시그널 한 개만 처리
    sig = signals[-1]
    logging.info(
        f"[PATTERN] {market} detected {len(signals)} patterns, "
        f"latest={sig.name}, dir={sig.direction}, target={sig.target}"
    )

    if within_cooldown(st.get('last_trade_at')):
        logging.info(f"[PATTERN] {market} {sig.name} ignored (cooldown)")
        return

    if sig.direction == 'up':
        if not enforce_risk_limits(market, CROSS_BUY_SIZE_KRW):
            logging.info(f"[PATTERN] {market} {sig.name} up but risk limit")
            return
        if get_balance('KRW') < CROSS_BUY_SIZE_KRW:
            logging.info(f"[PATTERN] {market} {sig.name} up but no KRW")
            return
        if split_buy_krw(market, CROSS_BUY_SIZE_KRW):
            st['last_side'] = 'long'
            st['last_trade_at'] = now_kst().isoformat()
            st['last_reason'] = f'pattern:{sig.name}'
            st['last_pattern'] = {
                'name': sig.name,
                'target': sig.target,
                'at': sig.breakout_ts.isoformat(),
            }
            tg_notify(f"[PATTERN BUY] {market} {sig.name} target={sig.target}")
    elif sig.direction == 'down':
        if get_coin_balance(market) <= 0:
            logging.info(f"[PATTERN] {market} {sig.name} down but no position")
            return
        if split_sell_ratio(market, CROSS_SELL_RATIO):
            st['last_side'] = 'flat'
            st['last_trade_at'] = now_kst().isoformat()
            st['last_reason'] = f'pattern:{sig.name}'
            st['last_pattern'] = {
                'name': sig.name,
                'target': sig.target,
                'at': sig.breakout_ts.isoformat(),
            }
            tg_notify(f"[PATTERN SELL] {market} {sig.name} target={sig.target}")

# =========================
# 5) DCA & 리포트
# =========================
def is_dca_time(ts: dt.datetime) -> bool:
    return ts.day == DCA_DAY and ts.hour == DCA_HOUR and ts.minute == DCA_MINUTE

def run_monthly_dca(tickers: List[str], state: Dict):
    if not DCA_ENABLED:
        return
    ts = now_kst()
    if not is_dca_time(ts):
        return
    ym = ts.strftime('%Y-%m')
    if state.get('_dca_last_ym') == ym:
        return

    logging.info(f"[DCA] monthly DCA start for {ym}")
    for mkt in tickers:
        if not enforce_risk_limits(mkt, DCA_AMOUNT_KRW):
            logging.info(f"[DCA] {mkt} skip (risk limit)")
            continue
        if get_balance('KRW') < DCA_AMOUNT_KRW:
            logging.info("[DCA] KRW 부족, DCA 중단")
            break
        logging.info(f"[DCA] {mkt} DCA buy {DCA_AMOUNT_KRW} KRW")
        split_buy_krw(mkt, DCA_AMOUNT_KRW)
        time.sleep(0.2)
    state['_dca_last_ym'] = ym
    save_state(state)
    tg_notify(f"[DCA] {ym} 월 정기 매수 완료")

def is_report_time(ts: dt.datetime) -> bool:
    return ts.hour == REPORT_HOUR and ts.minute == REPORT_MINUTE

def build_report_text() -> str:
    bal = UPBIT.get_balances()
    rows = []
    total_krw = 0.0
    for b in bal:
        cur = b.get('currency')
        amt = float(b.get('balance', 0.0))
        locked = float(b.get('locked', 0.0))
        if cur == 'KRW':
            total_krw += amt
            rows.append(f"- KRW: {amt:,.0f} (locked {locked})")
        else:
            market = f"KRW-{cur}"
            try:
                px = float(pyupbit.get_current_price(market) or 0.0)
            except Exception:
                px = 0.0
            val = (amt + locked) * px
            total_krw += val
            rows.append(f"- {cur}: {amt:.6f} @ {px:,.0f} ⇒ {val:,.0f} KRW")
    today = now_kst().date().isoformat()
    trades = []
    if os.path.exists(TRADELOG_CSV):
        df = pd.read_csv(TRADELOG_CSV)
        if not df.empty:
            df['ts'] = pd.to_datetime(df['ts'])
            df_kst = df.set_index('ts').tz_localize(None)
            df_today = df_kst[df_kst.index.date == now_kst().date()]
            for _, r in df_today.tail(20).iterrows():
                trades.append(f"{r['ts']} {r['side']} {r['market']} {r['value_or_qty']}")
    lines = [
        f"[Upbit Daily Report] {today}",
        "",
        "== Balances ==",
        *rows,
        "",
        f"총 평가액(대략): {total_krw:,.0f} KRW",
        "",
        "== Today's Trades ==",
        *(trades if trades else ["(no trades)"]),
    ]
    return "\n".join(lines)

def send_email_report():
    if not (SMTP_HOST and SMTP_USER and SMTP_PASS and REPORT_TO):
        logging.info("[REPORT] SMTP 설정 없음 → 스킵")
        return
    body = build_report_text()
    msg = MIMEMultipart()
    msg["Subject"] = f"Upbit Daily Report {now_kst().date().isoformat()}"
    msg["From"] = SMTP_USER
    msg["To"] = REPORT_TO
    msg.attach(MIMEText(body, "plain", "utf-8"))
    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as s:
            s.ehlo()
            s.starttls()
            s.login(SMTP_USER, SMTP_PASS)
            s.send_message(msg)
        logging.info("[REPORT] 이메일 전송 완료")
    except Exception as e:
        logging.exception(f"[REPORT-ERR] {e}")

# =========================
# 6) 메인 루프
# =========================
def graceful_exit(signum, frame):
    logging.info("신호 수신, 종료 중...")
    tg_notify("봇 종료 신호 수신")
    raise SystemExit

signal.signal(signal.SIGINT, graceful_exit)
signal.signal(signal.SIGTERM, graceful_exit)

def main_loop():
    state = load_state()
    ohlcv_cache: Dict[str, Tuple[pd.DataFrame, float]] = {}
    last_refetch = {m: 0 for m in TICKERS}
    last_report_ymd = None
    last_heartbeat = 0.0  # 헬스 체크 로깅용

    logging.info("=== Upbit Auto Trader 시작 ===")
    tg_notify("Upbit Auto Trader 시작")

    while True:
        try:
            # 월 1회 DCA
            run_monthly_dca(TICKERS, state)

            # 일일 리포트
            ts = now_kst()
            ymd = ts.strftime("%Y-%m-%d")
            if is_report_time(ts) and last_report_ymd != ymd:
                send_email_report()
                last_report_ymd = ymd

            # 1분에 한 번 헬스 체크 로그
            now_sec = time.time()
            if now_sec - last_heartbeat > 60:
                hb = {
                    mkt: state.get(mkt, {}).get('last_reason', 'none')
                    for mkt in TICKERS
                }
                logging.info(f"[HEARTBEAT] ts={ts.isoformat()} last_reason={hb}")
                last_heartbeat = now_sec

            # 각 종목 점검
            for mkt in TICKERS:
                now_sec = time.time()
                need_fetch = (now_sec - last_refetch[mkt] > REFETCH_EVERY_MIN * 60)
                if need_fetch or mkt not in ohlcv_cache:
                    df = fetch_ohlcv_all(mkt, LOOKBACK_DAYS)
                    ohlcv_cache[mkt] = (df, now_sec)
                    last_refetch[mkt] = now_sec
                else:
                    df = ohlcv_cache[mkt][0]

                # 빗각
                process_trend_signal(mkt, state, df)
                # 패턴
                process_pattern_signals(mkt, state, df)

                save_state(state)
                time.sleep(0.2)

            time.sleep(5)
        except SystemExit:
            break
        except Exception as e:
            logging.exception(f"[LOOP-ERR] {e}")
            time.sleep(3)

    logging.info("=== 종료 완료 ===")

# =========================
# 7) 백테스트(빗각 간단)
# =========================
def backtest_trend(market: str, days: int = 365*3, fee=0.0005) -> Dict:
    df = fetch_ohlcv_all(market, days)
    m, b, *_ = trendline_from_extrema(df)
    cash = 1_000_000.0
    coin = 0.0
    equity_curve = []
    for i in range(1, len(df)):
        prev_ts, cur_ts = df.index[i-1], df.index[i]
        prev_close, cur_close = float(df['close'].iloc[i-1]), float(df['close'].iloc[i])
        prev_line = projected_line_value(m, b, df, prev_ts)
        cur_line  = projected_line_value(m, b, df, cur_ts)
        cross = detect_cross(prev_close, prev_line, cur_close, cur_line)
        if cross == 'cross_up' and cash > 0:
            coin += (cash * (1 - fee)) / cur_close
            cash = 0.0
        elif cross == 'cross_down' and coin > 0:
            cash += coin * cur_close * (1 - fee)
            coin = 0.0
        equity = cash + coin * cur_close
        equity_curve.append((cur_ts, equity))
    if coin > 0:
        last_close = float(df['close'].iloc[-1])
        cash += coin * last_close * (1 - fee)
        coin = 0.0
    curve = pd.DataFrame(equity_curve, columns=['ts', 'equity']).set_index('ts')
    dd = (curve['equity'] / curve['equity'].cummax() - 1.0).min()
    return {"final_krw": cash, "max_dd": dd, "curve": curve}

if __name__ == "__main__":
    main_loop()
    # 백테스트 샘플:
    # res = backtest_trend("KRW-BTC", 365*3); print(res["final_krw"], res["max_dd"])

=== Upbit Auto Trader 시작 ===
=== Upbit Auto Trader 시작 ===
신호 수신, 종료 중...
신호 수신, 종료 중...
=== 종료 완료 ===
=== 종료 완료 ===
